## Loading Insurance Question Answering Dataset

In [96]:
from datasets import load_dataset
df=load_dataset("diya22/llm")

### Removing id column 

In [97]:
df = df.remove_columns(['id'])

### Dataset

In [98]:
df

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 800
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 200
    })
})

In [99]:
import torch
from transformers import Trainer, TrainingArguments

## Loading T5 Small Model 

In [100]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("google-t5/t5-small")
model = T5ForConditionalGeneration.from_pretrained("google-t5/t5-small")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


## Zero Short

In [101]:
# from transformers import pipeline
 
# # Initialize text generation pipeline
# generator = pipeline("question-answering", model="google-t5/t5-small",tokenizer='google-t5/t5-small')

In [102]:
from transformers import pipeline

# Load the question answering pipeline
# generator = pipeline("question-answering")
generator = pipeline("question-answering", model="google-t5/t5-small",tokenizer='google-t5/t5-small')
# Example question and context
question1 = "Is snow damage covered by insurance?"
context1 = "The answer is most often yes, if your home has a sloped roof, and your particular policy provides for that coverage."

# Format the input as a dictionary with 'question' and 'context' keys
input_dict = {"question": question1, "context": context1}

# Perform zero-shot inference using the question answering pipeline
zero_shot_answer = generator(input_dict, max_length=400, temperature=0, top_k=10)

print(zero_shot_answer)



Some weights of T5ForQuestionAnswering were not initialized from the model checkpoint at google-t5/t5-small and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[{'score': 0.002743419958278537, 'start': 14, 'end': 48, 'answer': 'most often yes, if your home has a'}, {'score': 0.0027111817616969347, 'start': 14, 'end': 37, 'answer': 'most often yes, if your'}, {'score': 0.0026650759391486645, 'start': 4, 'end': 48, 'answer': 'answer is most often yes, if your home has a'}, {'score': 0.0026337583549320698, 'start': 4, 'end': 37, 'answer': 'answer is most often yes, if your'}, {'score': 0.0025945224333554506, 'start': 11, 'end': 48, 'answer': 'is most often yes, if your home has a'}, {'score': 0.0025640339590609074, 'start': 11, 'end': 37, 'answer': 'is most often yes, if your'}, {'score': 0.002464889083057642, 'start': 14, 'end': 61, 'answer': 'most often yes, if your home has a sloped roof,'}, {'score': 0.0024519353173673153, 'start': 14, 'end': 55, 'answer': 'most often yes, if your home has a sloped'}, {'score': 0.002381914993748069, 'start': 4, 'end': 55, 'answer': 'answer is most often yes, if your home has a sloped'}, {'score': 0.002345052

## One Short

In [103]:
from transformers import pipeline

# Load the question answering pipeline
# generator = pipeline("question-answering")
generator = pipeline("question-answering", model="google-t5/t5-small",tokenizer='google-t5/t5-small')
# Example question and context
question2 = "Is snow damage covered by insurance?"
context2 = "Great question! The answer is most often yes, if your home has a sloped roof, and your particular policy provides for that coverage. If your roof is flat, that is probably a different story. Typically with a flat roof, the risk for ice or snow damage is greatly increased, so your policy is either much more expensive, or that damage is excluded. Call your agent to be certain in either case. Good luck! Thanks for asking!"

# Format the input as a dictionary with 'question' and 'context' keys
input_dict = {"question": question2, "context": context2}

# Perform zero-shot inference using the question answering pipeline
one_shot_answer = generator(input_dict, max_length=400, temperature=0, top_k=10)

print(one_shot_answer)


Some weights of T5ForQuestionAnswering were not initialized from the model checkpoint at google-t5/t5-small and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[{'score': 0.00021120022574905306, 'start': 146, 'end': 200, 'answer': 'is flat, that is probably a different story. Typically'}, {'score': 0.00020889114239253104, 'start': 174, 'end': 200, 'answer': 'different story. Typically'}, {'score': 0.0002069838810712099, 'start': 172, 'end': 200, 'answer': 'a different story. Typically'}, {'score': 0.00020685135677922517, 'start': 149, 'end': 200, 'answer': 'flat, that is probably a different story. Typically'}, {'score': 0.00020650078658945858, 'start': 160, 'end': 200, 'answer': 'is probably a different story. Typically'}, {'score': 0.0002033735072473064, 'start': 163, 'end': 200, 'answer': 'probably a different story. Typically'}, {'score': 0.00020317049347795546, 'start': 298, 'end': 351, 'answer': 'much more expensive, or that damage is excluded. Call'}, {'score': 0.00020107808813918382, 'start': 337, 'end': 397, 'answer': 'excluded. Call your agent to be certain in either case. Good'}, {'score': 0.00019771608640439808, 'start': 146, 'end

## Few Shorts

In [104]:
from transformers import pipeline

# Load the question answering pipeline
# generator = pipeline("question-answering")
generator = pipeline("question-answering", model="google-t5/t5-small",tokenizer='google-t5/t5-small')
# Example question and context
# Define the questions and contexts
# Define the questions and contexts
questions = [
    "Is snow damage covered by insurance?",
    "Will Life Insurance Know If I Smoke?",
    "Does Blue Cross Blue Shield Have Life Insurance?"
]

contexts = [
    "Great question! The answer is most often yes, if your home has a sloped roof, and your particular policy provides for that coverage...",
    "That is a great question! The answer is simple, the cost of your auto insurance for your imported car is higher because the cost to repair it...",
    "Blue Cross / Blue Shield is the name of the network association for a number of health insurance companies (includes Anthem and CareFirst)..."
]

# Perform few-shot inference for each question and context
for i in range(len(questions)):
    # Call the generator pipeline with each question and context pair
    answer = generator(question=questions[i], context=contexts[i], max_length=400, temperature=0, top_k=10)
    
    # Print the question and its answer
    print(f"Question: {questions[i]}")
    # print(answer)
    print(f"Answer: {answer[3]}\n")

Some weights of T5ForQuestionAnswering were not initialized from the model checkpoint at google-t5/t5-small and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Question: Is snow damage covered by insurance?
Answer: {'score': 0.0017730311956256628, 'start': 65, 'end': 117, 'answer': 'sloped roof, and your particular policy provides for'}

Question: Will Life Insurance Know If I Smoke?
Answer: {'score': 0.0017919926904141903, 'start': 80, 'end': 123, 'answer': 'for your imported car is higher because the'}

Question: Does Blue Cross Blue Shield Have Life Insurance?
Answer: {'score': 0.0016422035405412316, 'start': 52, 'end': 69, 'answer': 'association for a'}



## Tokenization Function

In [106]:
# Tokenize and encode your input and output sequences
def preprocess_function(examples):
    inputs = tokenizer(examples['question'], truncation=True,padding=True,max_length=512)
    outputs = tokenizer(examples['answer'],  truncation=True,padding=True,max_length=512)
    # # Update examples with inputs and outputs
    examples["input_ids"] = inputs.input_ids
    examples["attention_mask"] = inputs.attention_mask
    examples["labels"] = outputs.input_ids
    # We need to return the examples dictionary
    return examples

### Applying tokenization function to train and test dataset

In [107]:
train_dataset = df['train'].map(preprocess_function,batched=True).select(range(10))
test_dataset = df['test'].map(preprocess_function,batched=True).select(range(10))
# # train_dataset=df['train']
# train_dataset=df['train'].map(preprocess_function,batched=True)

### Train Dataset

In [108]:
train_dataset

Dataset({
    features: ['question', 'answer', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 10
})

## Test Dataset

In [109]:
test_dataset

Dataset({
    features: ['question', 'answer', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 10
})

## Removing question, answer and attention mask column 

In [110]:
train_dataset = train_dataset.remove_columns(['question','answer','attention_mask'])
test_dataset = test_dataset.remove_columns(['question','answer',"attention_mask"])

In [111]:
train_dataset

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 10
})

## Giving training Arguments

In [112]:
from transformers import Trainer, TrainingArguments
output_dir = 'checkpoints'
 
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    evaluation_strategy="epoch"
)

## Loading metric for checking accuracy

In [113]:
import numpy as np
import evaluate
 
metric = evaluate.load("accuracy")

## Compute metrics

In [114]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

## Trainer

In [115]:
from transformers import Trainer

 
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics)

## Model Training

In [116]:
trainer.train()

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (2, 10) + inhomogeneous part.

In [ ]:
# train_dataset[11]['question']

'Who  Should  I  Set  Up  A  Roth  IRA  With?'

In [ ]:
# train_dataset[11]['answer']

'It depends on which Province you live in. In Ontario, you don\'t pay a premium for basic healthcare but in British Columbia you do. We moved away from Canada because the taxes were so high so "free healthcare" is up for debate. Also, Doctors can be hard to find in some areas.'

In [ ]:
# from transformers import pipeline
# qa=pipeline('question-answering',model='google-t5/t5-small',tokenizer='google-t5/t5-small')

Some weights of T5ForQuestionAnswering were not initialized from the model checkpoint at google-t5/t5-small and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# output=qa(train_dataset[11]['question'],train_dataset[11]['answer'])
# print(output)

{'score': 0.0007245090091601014, 'start': 68, 'end': 69, 'answer': 'a'}


In [ ]:
# input_ids=tokenizer('Povide answer to this question: How To Choose The Best Auto Insurance?',return_tensors='pt').input_ids

In [ ]:
# generated_ids=model.generate(input_ids=input_ids)

In [ ]:
# preds=[ for gen_ids in generated_ids:  tokenizer.decode(gen_ids,skip_special_tokens=True,clean_up_tokenization_spaces=True)]